# Notebook ٦: Tokenization (تقطيع النص)

BPE وWordPiece على أسماء لاعبين حقيقية ونص عربي. tokenizers حقيقية من Hugging Face، كل خلية اتشغلت فعلًا.

## ١. التقطيع بالمسافات ومشكلة الكلمات الجديدة

لو الـ vocab كلمات كاملة، أي كلمة جديدة بتبقى مجهولة (UNK).

In [ ]:
# جمل تدريب بسيطة كقايمة كلمات
train = "salah scored a goal and salah scored again".split()
# الـ vocabulary: الكلمات الفريدة
vocab = sorted(set(train))
# نطبعه
print(vocab)
# جملة فيها كلمات مش في التدريب
new = "salah scores goals".split()
# كل كلمة مش معروفة بتبقى unk
print([w if w in vocab else "<unk>" for w in new])


## ٢. BPE من الصفر

بنبدأ بحروف، وبندمج أكتر زوج حروف بيتكرر مرة بعد مرة.

In [ ]:
# عدّاد
from collections import Counter
# كلمات التدريب
words = ["salah", "salah", "salah", "scored", "scores", "score", "goal", "goals"]
# كل كلمة حروف + علامة نهاية
seqs = [list(w) + ["_"] for w in words]
# أكتر زوج متجاور تكرارًا
def best_pair(seqs):
# عدّاد الأزواج
    c = Counter()
# لكل كلمة
    for s in seqs:
# لكل زوج متجاور
        for a, b in zip(s, s[1:]):
# نزود عدّاد الزوج
            c[(a, b)] += 1
# نرجع الأكتر تكرارًا
    return c.most_common(1)[0]
# تدمج الزوج ده في كل الكلمات
def merge(seqs, pair):
# القايمة الجديدة
    out = []
# لكل كلمة
    for s in seqs:
# مؤشر وكلمة جديدة
        i, n = 0, []
# نمشي على الحروف
        while i < len(s):
# لو لقينا الزوج
            if i < len(s) - 1 and (s[i], s[i + 1]) == pair:
# ندمجهم
                n.append(s[i] + s[i + 1]); i += 2
# غير كده
            else:
# ننسخ الحرف
                n.append(s[i]); i += 1
# نضيف الكلمة
        out.append(n)
# نرجع القايمة
    return out
# 8 عمليات دمج
for step in range(8):
# الزوج الأكتر تكرارًا
    pair, cnt = best_pair(seqs)
# ندمجه
    seqs = merge(seqs, pair)
# نطبع الدمج وعدد مرات تكراره
    print(step + 1, pair, cnt)
# شكل salah وscored وgoal بعد الدمج
print(seqs[0], seqs[3], seqs[6])


## ٣. WordPiece الحقيقي (BERT)

الكلمة الغريبة بتتقسم لأجزاء معروفة، والأجزاء الداخلية بتبدأ بـ ##.

In [ ]:
# نخفي التحذيرات
import warnings; warnings.filterwarnings("ignore")
# تحميل الـ tokenizers
from transformers import AutoTokenizer
# tokenizer نموذج BERT
bert = AutoTokenizer.from_pretrained("bert-base-uncased")
# حجم الـ vocabulary
print(bert.vocab_size)
# كلمات للتجربة
for w in ["Salah", "Haaland", "Odegaard", "unbelievable"]:
# نقسمها لأجزاء
    print(w, "->", bert.tokenize(w))


In [ ]:
# نشفّر جملتين بحشو
enc = bert(["is salah injured", "next match"], padding=True, return_tensors="np")
# التوكنز مع الرموز الخاصة
print(bert.convert_ids_to_tokens(enc["input_ids"][0]))
# الأرقام (الجملة الأقصر اتحشت بصفر)
print(enc["input_ids"])
# الماسك: 1 لحقيقي و0 لحشو
print(enc["attention_mask"])


## ٤. GPT-2 BPE

الـ Ġ معناها إن الكلمة بدأت بمسافة.

In [ ]:
# tokenizer نموذج GPT-2
gpt = AutoTokenizer.from_pretrained("gpt2")
# حجم الـ vocab
print(gpt.vocab_size)
# تقطيع جملة
print(gpt.tokenize("Mohamed Salah scored twice"))
# عدد التوكنز
print(len(gpt("Mohamed Salah scored twice")["input_ids"]))


## ٥. العربي: التوكنز ليه تكلفة

نفس المعنى بالعربي ممكن ياخد توكنز أكتر، وده بيأثر على التكلفة وطول الـ context.

In [ ]:
# tokenizer متعدد اللغات
mbert = AutoTokenizer.from_pretrained("bert-base-multilingual-cased")
# جملة عربية
ar = "محمد صلاح سجل هدفين في المباراة"
# نفس المعنى بالإنجليزي
en = "Mohamed Salah scored two goals in the match"
# BERT الإنجليزي: حرف حرف
print(bert.tokenize(ar))
# البديل المتعدد اللغات: كلمات
print(mbert.tokenize(ar))
# نقارن 3 tokenizers
for name, t in [("bert-en", bert), ("mbert", mbert), ("gpt2", gpt)]:
# عدد التوكنز لكل لغة
    print(f"{name:8} arabic={len(t.tokenize(ar)):2} english={len(t.tokenize(en)):2}")


تنبيه: الحروف العربية جوه الـ output ممكن تظهر مرتبة معكوسة بسبب اتجاه الكتابة في الـ PDF، الشغل الحقيقي في الأرقام: 26 توكن مقابل 7.

## ٦. أسماء لاعبين حقيقية من FPL

أسماء الناس والأندية بتتقسم أكتر من الكلمات العادية.

In [ ]:
# قراءة وجداول
import json, pandas as pd
# أسماء اللاعبين من FPL
names = pd.DataFrame(json.load(open("fpl.json"))["elements"])["web_name"]
# عدد توكنز كل اسم
n_tok = names.apply(lambda s: len(bert.tokenize(s)))
# نسبة الأسماء المتقسمة
print(round((n_tok > 1).mean() * 100, 1), "% of names split into 2+ tokens")
# توزيع عدد التوكنز
print(n_tok.value_counts().sort_index().to_dict())
# أسماء اتقسمت لـ 4 توكنز أو أكتر
print(names[n_tok >= 4].head(5).tolist())
